In [1]:
import tensorflow as tf 
from tensorflow import keras 
import tensorflow_addons as tfa 
import pandas as pd
import numpy as np 
from sklearn.metrics import mean_absolute_error
from models import load_ef_model

c:\Users\SIA\anaconda3\envs\deepface-env\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


### Load EarlyFusion model

In [2]:
ef_model = load_ef_model()
ef_model.summary()

Model: "ef_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 Scene_input (InputLayer)       [(None, 10, 224, 22  0           []                               
                                4, 3)]                                                            
                                                                                                  
 Face_input (InputLayer)        [(None, 10, 224, 22  0           []                               
                                4, 3)]                                                            
                                                                                                  
 Audio_input (InputLayer)       [(None, 15, 128)]    0           []                               
                                                                                           

### Load data

In [3]:
AUTOTUNE = tf.data.AUTOTUNE

# Train
scene_train_ds = tf.data.experimental.load('./data/fullscene/train_ds/')
face_train_ds  = tf.data.experimental.load('./data/faces/train_ds/')
audio_train_ds = tf.data.experimental.load('./data/audio/train_ds/')
text_train_ds  = tf.data.experimental.load('./data/text/train_ds/').batch(batch_size=32)

scene_xtrain = scene_train_ds.map(lambda x,y: x)
face_xtrain  = face_train_ds.map(lambda x,y: x)
audio_xtrain = audio_train_ds.map(lambda x,y: x)
text__xtrain = text_train_ds.map(lambda x,y: x)
y_train      = scene_train_ds.map(lambda x,y: y)

train_ds = tf.data.Dataset.zip(((scene_xtrain, face_xtrain, audio_xtrain, text__xtrain), y_train)).shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)


# Valid
scene_valid_ds = tf.data.experimental.load('./data/fullscene/val_ds/')
face_valid_ds  = tf.data.experimental.load('./data/faces/val_ds/')
audio_valid_ds = tf.data.experimental.load('./data/audio/val_ds') 
text_valid_ds  = tf.data.experimental.load('./data/text/val_ds/').batch(batch_size=32)

scene_xvalid = scene_valid_ds.map(lambda x,y: x)
face_xvalid  = face_valid_ds.map(lambda x,y: x)
audio_xvalid = audio_valid_ds.map(lambda x,y: x)
text_xvalid  = text_valid_ds.map(lambda x,y: x)
y_valid      = scene_valid_ds.map(lambda x,y: y)

valid_ds = tf.data.Dataset.zip(((scene_xvalid, face_xvalid, audio_xvalid, text_xvalid), y_valid)).shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)

train_ds, valid_ds

Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.


Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.


(<_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 50), dtype=tf.int32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>,
 <_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 50), dtype=tf.int32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>)

### Compile and Train model

In [4]:
import datetime
t = datetime.datetime.now().strftime("%m%d_%H%M%S")

early_stopping = keras.callbacks.EarlyStopping(patience=10, verbose=0)
check_point    = keras.callbacks.ModelCheckpoint(filepath='./weights/ef/ef.t5',
                             monitor='val_mae',
                             mode='min',
                             save_best_only=True,
                             save_weights_only=True,
                             verbose=0)

optimizer = tfa.optimizers.RectifiedAdam()
ef_model.compile(loss='mse', optimizer=optimizer, metrics=['mae'])
history = ef_model.fit(train_ds, validation_data=valid_ds, batch_size=32, epochs=100, callbacks=[early_stopping, check_point])

Epoch 1/100
1/1 [==============================] - 94s 94s/step - loss: 0.0030 - mae: 0.0439 - val_loss: 0.0014 - val_mae: 0.0305
Epoch 2/100
1/1 [==============================] - 29s 29s/step - loss: 0.0042 - mae: 0.0559 - val_loss: 0.0014 - val_mae: 0.0305
Epoch 3/100
1/1 [==============================] - 29s 29s/step - loss: 0.0038 - mae: 0.0539 - val_loss: 0.0014 - val_mae: 0.0305
Epoch 4/100
1/1 [==============================] - 49s 49s/step - loss: 0.0042 - mae: 0.0561 - val_loss: 0.0014 - val_mae: 0.0305
Epoch 5/100
1/1 [==============================] - 34s 34s/step - loss: 0.0043 - mae: 0.0563 - val_loss: 0.0014 - val_mae: 0.0305
Epoch 6/100
1/1 [==============================] - 42s 42s/step - loss: 0.0043 - mae: 0.0559 - val_loss: 0.0013 - val_mae: 0.0310
Epoch 7/100
1/1 [==============================] - 22s 22s/step - loss: 0.0046 - mae: 0.0587 - val_loss: 0.0013 - val_mae: 0.0317
Epoch 8/100
1/1 [==============================] - 42s 42s/step - loss: 0.0029 - mae: 0.04

## Evaluation

### Validation data

In [5]:
AUTOTUNE = tf.data.AUTOTUNE
scene_valid_ds = tf.data.experimental.load('./data/fullscene/val_ds/')
face_valid_ds  = tf.data.experimental.load('./data/faces/val_ds/')
audio_valid_ds = tf.data.experimental.load('./data/audio/val_ds') 
text_valid_ds  = tf.data.experimental.load('./data/text/val_ds/').batch(batch_size=32)

scene_xvalid = scene_valid_ds.map(lambda x,y: x)
face_xvalid  = face_valid_ds.map(lambda x,y: x)
audio_xvalid = audio_valid_ds.map(lambda x,y: x)
text_xvalid  = text_valid_ds.map(lambda x,y: x)
y_valid      = scene_valid_ds.map(lambda x,y: y)

valid_ds = tf.data.Dataset.zip(((scene_xvalid, face_xvalid, audio_xvalid, text_xvalid), y_valid)).prefetch(buffer_size=AUTOTUNE)


### Load weights

In [6]:
ef_model.load_weights('./weights/ef/ef.t5')
loss, mae = ef_model.evaluate(valid_ds)
(1-mae)*100

1/1 [==============================] - 7s 7s/step - loss: 0.0014 - mae: 0.0305


96.95181008428335

### Validation data

In [7]:
y_true = np.concatenate([y for x,y in valid_ds], axis=0)
y_pred = ef_model.predict(valid_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 16s 16s/step


(array([95.32583 , 98.973305, 99.97168 , 96.20932 , 94.27892 ],
       dtype=float32),
 96.95181008428335)

### Test data

In [8]:
scene_test_ds = tf.data.experimental.load('./data/fullscene/test_ds/')
face_test_ds  = tf.data.experimental.load('./data/faces/test_ds/')
audio_test_ds = tf.data.experimental.load('./data/audio/test_ds') 
text_test_ds  = tf.data.experimental.load('./data/text/test_ds/').batch(batch_size=32)


scene_xtest = scene_test_ds.map(lambda x,y: x)
face_xtest  = face_test_ds.map(lambda x,y: x)
audio_xtest = audio_test_ds.map(lambda x,y: x)
text_xtest  = text_test_ds.map(lambda x,y: x)

y_test      = scene_test_ds.map(lambda x,y: y)

test_ds = tf.data.Dataset.zip(((scene_xtest, face_xtest, audio_xtest, text_xtest), y_test)).prefetch(buffer_size=AUTOTUNE)

test_ds

<_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 50), dtype=tf.int32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>

In [9]:
with tf.device('/gpu:0'):
    loss, mae = ef_model.evaluate(test_ds)
(1-mae)*100

1/1 [==============================] - 8s 8s/step - loss: 0.0064 - mae: 0.0682


93.18354353308678

In [10]:
y_true = np.concatenate([y for x,y in test_ds], axis=0)
y_pred = ef_model.predict(test_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 5s 5s/step


(array([88.842606, 87.74629 , 95.81683 , 98.89607 , 94.61591 ],
       dtype=float32),
 93.18354353308678)